# Analyse des erreurs et de leur disponibilité temporelle

Diagnostics **2023 / 2024 / 2025**, sans amélioration implémentée. Model B — full hierarchical (`P1_G1`) reste la représentation principale, pas un champion.

Toutes les tables sont des marginales agrégées. Aucun identifiant d'unité, dossier CRM ou CSV n'est affiché/exporté. Statistiques numériques masquées sous cinq observations ; modalités rares regroupées. Rapport détaillé : [analyse des erreurs](../docs/error_forensics.md).

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Exécuter depuis le dépôt ou notebooks/.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
leases = pd.read_csv(ROOT / 'data/raw/equinoxe_lease_history.csv',
                     dtype={'sPropCode': 'string', 'sUnitCode': 'string'})
# Les baux et les vecteurs unitaires restent en mémoire, sans affichage ni export.

from src.evaluation.error_forensics import run_forensics
external = pd.read_csv(ROOT / 'data/external/external_signals.csv')
from src.external_data import validate_external_data
assert not validate_external_data(external), 'Schéma externe existant non conforme.'
findings = run_forensics(leases, external)
# Contrôle des tables affichables : aucun champ identifiant d'unité ou bail.
for name, table in findings.items():
    assert not {'sUnitCode', 'sPropCode', 'UNIT_KEY', 'hTenant', 'hLease'}.intersection(table.columns), name


## Composition à chaque origine et changements de mix

Cutoffs : 2022-12-31, 2023-12-31, 2024-12-31. Les cohortes précédentes sont reconstruites à leurs propres origines. La première présence dans l'extrait n'est pas une date d'ouverture d'immeuble.

Ces caractéristiques sont admissibles **selon le contrat**, mais strictement `UNKNOWN` en l'absence de versions CRM archivées. Cette réserve n'autorise aucun nouveau champ dans un modèle.

In [2]:
display(findings['building_presence'].round(4))
display(findings['mix_changes'].round(6))
display(findings['numeric'].round(4))
display(findings['categories'].round(4))

,year,building,first_start_in_known_extract,known_lease_count,candidate_units,share_pct,previous_candidates,previous_share_pct,new_to_candidate_cohort
0,2023,Daniel-Johnson,2019-01-01,282,111,27.4074,53,15.1429,False
1,2023,Levesque,2018-01-01,397,81,20.0000,83,23.7143,False
2,2023,Saint-Elzear,2017-01-01,1070,213,52.5926,214,61.1429,False
3,2024,Daniel-Johnson,2019-01-01,416,125,19.3798,111,27.4074,False
4,2024,Le Carlyle,2023-01-01,155,127,19.6899,0,0.0000,True
5,2024,Levesque,2018-01-01,480,82,12.7132,81,20.0000,False
6,2024,Saint-Elzear,2017-01-01,1287,215,33.3333,213,52.5926,False
7,2024,The Met,2023-01-01,104,96,14.8837,0,0.0000,True
8,2025,Daniel-Johnson,2019-01-01,552,129,15.0701,125,19.3798,False
9,2025,Le Carlyle,2023-01-01,350,180,21.0280,127,19.6899,False


,year,feature,total_variation,current_units,previous_units
0,2023,building,0.122646,405,350
1,2023,province,0.000000,405,350
2,2023,bedrooms,0.051993,405,350
3,2023,floor,0.051429,405,350
4,2023,unit_subtype,0.051993,405,350
5,2023,expiry_month,0.071570,405,350
6,2024,building,0.345736,645,405
7,2024,province,0.148837,645,405
8,2024,bedrooms,0.098306,645,405
9,2024,floor,0.132300,645,405


,year,feature,available,missing,median,previous_median,q25,q75
0,2023,sqft,405,0,1165.0000,1182.5000,755.0000,1295.0000
1,2023,floor,405,0,10.0000,11.0000,6.0000,16.0000
2,2023,prior_contractual_rent,405,0,1930.0000,1850.0000,1405.0000,2225.0000
3,2023,prior_term_months,405,0,12.0000,12.0000,12.0000,12.0000
4,2023,months_since_known_start,405,0,5.4538,5.2402,3.0226,7.8850
5,2024,sqft,645,0,1105.0000,1165.0000,660.0000,1275.0000
6,2024,floor,645,0,9.0000,10.0000,5.0000,16.0000
7,2024,prior_contractual_rent,645,0,2100.0000,1930.0000,1465.0000,2455.0000
8,2024,prior_term_months,645,0,12.0000,12.0000,12.0000,12.0000
9,2024,months_since_known_start,645,0,5.5852,5.4538,3.5154,7.7864


,year,feature,category,units,share_pct
0,2023,building,Saint-Elzear,213,52.5926
1,2023,building,Daniel-Johnson,111,27.4074
2,2023,building,Levesque,81,20.0000
3,2023,province,Quebec,405,100.0000
4,2023,bedrooms,2,287,70.8642
...,...,...,...,...,...
150,2025,expiry_month,10,57,6.6589
151,2025,expiry_month,3,55,6.4252
152,2025,expiry_month,11,54,6.3084
153,2025,expiry_month,2,48,5.6075


## Historique qualifié, support et maturité

Les labels en attente restent inconnus et ne sont pas imputés. Les fenêtres récentes sont descriptives et fixées ; aucun choix de fenêtre n'est entraîné sur ces résultats. A utilise des unités/années positives ; B des transitions mûres.

In [3]:
display(findings['historical_segments'].round(6))

,year,dimension,segment,qualified_occurrence,pending_labels,historical_occurrence_pct,qualified_positive_unit_years,historical_unit_growth_median_pct,historical_unit_growth_q25_pct,historical_unit_growth_q75_pct,qualified_growth_transitions,historical_transition_growth_median_pct,growth_q25_pct,growth_q75_pct
0,2023,building,Daniel-Johnson,97.0,56.0,98.969072,96.0,3.400080,0.938902,5.138704,NaN,NaN,NaN,NaN
1,2023,building,Levesque,224.0,80.0,98.660714,221.0,2.165960,-0.324897,5.140152,NaN,NaN,NaN,NaN
2,2023,building,Saint-Elzear,576.0,215.0,98.090278,565.0,2.919799,0.345872,5.182295,NaN,NaN,NaN,NaN
3,2023,province,Quebec,897.0,351.0,98.327759,882.0,2.918870,0.190677,5.172741,NaN,NaN,NaN,NaN
4,2023,forecast_year,2018,80.0,0.0,100.000000,80.0,2.019638,0.586202,3.869157,NaN,NaN,NaN,NaN
5,2023,forecast_year,2019,156.0,0.0,99.358974,155.0,1.614017,-1.215934,4.300739,NaN,NaN,NaN,NaN
6,2023,forecast_year,2020,307.0,0.0,99.022801,304.0,3.143685,0.200762,5.059922,NaN,NaN,NaN,NaN
7,2023,forecast_year,2021,344.0,11.0,98.255814,338.0,3.545463,0.639379,5.853900,NaN,NaN,NaN,NaN
8,2023,forecast_year,2022,10.0,340.0,50.000000,5.0,1.130435,-2.274352,2.264068,NaN,NaN,NaN,NaN
9,2023,B_growth_province,Quebec,NaN,NaN,NaN,NaN,NaN,NaN,NaN,944.0,2.881553,NaN,NaN


## Résultats ex post : bâtiments, provinces et autres segments

**RETROSPECTIVE_ONLY** : taux réalisés, croissances et sensibilités n'entrent jamais dans X. P1 par bâtiment n'est pas une contribution additive à la médiane globale. Retirer un bâtiment puis recalculer P1 est une sensibilité, pas une attribution causale. L'erreur de nombre attendu d'occurrences, somme de p moins somme de O, est additive.

In [4]:
segments = findings['realized_segments']
buildings = segments.loc[segments.dimension.eq('building')].copy()
display(buildings.round(6))
display(segments.loc[segments.dimension.eq('province')].round(6))
display(segments.loc[segments.dimension.isin(['bedrooms','floor','unit_subtype','expiry_month'])].round(6))

,year,dimension,segment,candidate_units,share_pct,observed_units,non_transitions,realized_occurrence_pct,realized_conditional_growth_pct,realized_segment_P1_pct,...,A_occurrence_pct,A_growth_pct,A_segment_P1_pct,A_occurrence_error_count,P1_G1_occurrence_pct,P1_G1_growth_pct,P1_G1_segment_P1_pct,P1_G1_occurrence_error_count,P1_without_building_pct,leave_out_sensitivity_pp
0,2023,building,Daniel-Johnson,111,27.407407,110,1,99.099099,2.331365,2.300443,...,98.327759,2.918870,2.870060,-0.856187,99.295177,2.881567,2.870976,0.217646,1.994340,-0.041762
1,2023,building,Levesque,81,20.000000,81,0,100.000000,2.096811,2.096811,...,98.327759,2.918870,2.870060,-1.354515,98.307685,2.881567,2.872388,-1.370775,1.995865,-0.040237
2,2023,building,Saint-Elzear,213,52.592593,210,3,98.591549,1.969006,1.879025,...,98.327759,2.918870,2.870060,-0.561873,98.261061,2.881567,2.871314,-0.703941,2.144614,0.108511
47,2024,building,Le Carlyle,127,19.689922,124,3,97.637795,1.539694,1.473402,...,98.475120,3.319802,3.269179,1.063403,98.751629,3.244485,3.236754,1.414568,1.708589,0.100737
48,2024,building,Daniel-Johnson,125,19.379845,123,2,98.400000,1.939124,1.738363,...,98.475120,3.319802,3.269179,0.093900,98.672586,3.244485,3.236099,0.340733,1.606464,-0.001388
49,2024,building,Levesque,82,12.713178,82,0,100.000000,2.487170,2.487170,...,98.475120,3.319802,3.269179,-1.250401,97.794985,3.244485,3.236989,-1.808113,1.501974,-0.105879
50,2024,building,The Met,96,14.883721,96,0,100.000000,1.745485,1.745485,...,98.475120,3.319802,3.269179,-1.463884,98.475120,3.255591,3.205947,-1.463884,1.550593,-0.057259
51,2024,building,Saint-Elzear,215,33.333333,209,6,97.209302,1.457401,1.302539,...,98.475120,3.319802,3.269179,2.721509,98.463813,3.244485,3.236754,2.697197,1.699619,0.091767
100,2025,building,Le Carlyle,180,21.028037,176,4,97.777778,3.793981,3.677122,...,98.189499,3.024079,2.969328,0.741098,97.299517,2.931619,2.924144,-0.860870,3.181530,-0.119348
101,2025,building,Daniel-Johnson,129,15.070093,126,3,97.674419,3.884338,3.819573,...,98.189499,3.024079,2.969328,0.664454,97.847920,2.931619,2.924144,0.223817,3.252007,-0.048872


,year,dimension,segment,candidate_units,share_pct,observed_units,non_transitions,realized_occurrence_pct,realized_conditional_growth_pct,realized_segment_P1_pct,...,A_occurrence_pct,A_growth_pct,A_segment_P1_pct,A_occurrence_error_count,P1_G1_occurrence_pct,P1_G1_growth_pct,P1_G1_segment_P1_pct,P1_G1_occurrence_error_count,P1_without_building_pct,leave_out_sensitivity_pp
3,2023,province,Quebec,405,100.000000,401,4,99.012346,2.085372,2.036102,...,98.327759,2.918870,2.870060,-2.772575,98.553810,2.881567,2.871314,-1.857070,NaN,NaN
52,2024,province,Quebec,549,85.116279,538,11,97.996357,1.685771,1.550593,...,98.475120,3.319802,3.269179,2.628411,98.478030,3.244485,3.236754,2.644386,NaN,NaN
53,2024,province,Ontario,96,14.883721,96,0,100.000000,1.745485,1.745485,...,98.475120,3.319802,3.269179,-1.463884,98.475120,3.255591,3.205947,-1.463884,NaN,NaN
106,2025,province,Quebec,736,85.981308,655,81,88.994565,3.782138,3.288111,...,98.189499,3.024079,2.969328,67.674713,97.753473,2.931619,2.924196,64.465561,NaN,NaN
107,2025,province,Ontario,120,14.018692,118,2,98.333333,3.388412,3.366310,...,98.189499,3.024079,2.969328,-0.172601,98.189499,2.948542,2.895159,-0.172601,NaN,NaN


,year,dimension,segment,candidate_units,share_pct,observed_units,non_transitions,realized_occurrence_pct,realized_conditional_growth_pct,realized_segment_P1_pct,...,A_occurrence_pct,A_growth_pct,A_segment_P1_pct,A_occurrence_error_count,P1_G1_occurrence_pct,P1_G1_growth_pct,P1_G1_segment_P1_pct,P1_G1_occurrence_error_count,P1_without_building_pct,leave_out_sensitivity_pp
4,2023,bedrooms,1,100,24.691358,99,1,99.000000,2.213921,2.191302,...,98.327759,2.918870,2.870060,-0.672241,98.447297,2.881567,2.871314,-0.552703,NaN,NaN
5,2023,bedrooms,2,287,70.864198,284,3,98.954704,1.871051,1.828939,...,98.327759,2.918870,2.870060,-1.799331,98.644677,2.881567,2.871314,-0.889777,NaN,NaN
6,2023,bedrooms,3,9,2.222222,9,0,100.000000,5.045441,5.045441,...,98.327759,2.918870,2.870060,-0.150502,95.739590,2.881567,2.872388,-0.383437,NaN,NaN
7,2023,bedrooms,0,9,2.222222,9,0,100.000000,1.946871,1.946871,...,98.327759,2.918870,2.870060,-0.150502,99.653847,2.881567,2.871314,-0.031154,NaN,NaN
8,2023,floor,1,5,1.234568,5,0,100.000000,-4.335447,-4.335447,...,98.327759,2.918870,2.870060,-0.083612,99.663735,2.881567,2.871314,-0.016813,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150,2025,expiry_month,7,85,9.929907,80,5,94.117647,3.283838,3.177704,...,98.189499,3.024079,2.969328,3.461074,99.597908,2.931619,2.927286,4.658222,NaN,NaN
151,2025,expiry_month,9,91,10.630841,84,7,92.307692,3.342272,3.028213,...,98.189499,3.024079,2.969328,5.352444,98.762164,2.931619,2.898644,5.873570,NaN,NaN
152,2025,expiry_month,2,48,5.607477,45,3,93.750000,3.494985,3.366675,...,98.189499,3.024079,2.969328,2.130960,99.459987,2.931619,2.921109,2.740794,NaN,NaN
153,2025,expiry_month,11,54,6.308411,41,13,75.925926,3.067339,1.284855,...,98.189499,3.024079,2.969328,12.022330,91.491117,2.931619,2.652924,8.405203,NaN,NaN


## Substitution rétrospective de l'occurrence et couverture

Conserver g prévu et remplacer p par O réalisé sert seulement à mesurer ce qui reste de l'écart. Ce calcul consulte le futur et **n'est jamais un forecast**. Les moyennes par unité sont conservées lorsqu'il existe plusieurs transitions.

In [5]:
display(findings['retrospective_components'].round(6))
display(findings['label_coverage'])

,year,model,predicted_P1_pct,P1_with_realized_occurrence_pct,realized_P1_pct,occurrence_replacement_change_pp,remaining_growth_and_distribution_gap_pp,availability_class
0,2023,A,2.870060,2.918870,2.036102,0.048811,0.882768,RETROSPECTIVE_ONLY
1,2023,P1_G1,2.871314,2.881567,2.036102,0.010252,0.845464,RETROSPECTIVE_ONLY
2,2024,A,3.269179,3.319802,1.607852,0.050623,1.711949,RETROSPECTIVE_ONLY
3,2024,P1_G1,3.236099,3.244485,1.607852,0.008386,1.636632,RETROSPECTIVE_ONLY
4,2025,A,2.969328,3.024079,3.300878,0.054751,-0.276800,RETROSPECTIVE_ONLY
5,2025,P1_G1,2.924144,2.931619,3.300878,0.007476,-0.369259,RETROSPECTIVE_ONLY


,year,candidate_units,annual_pairs_in_cohort,eligible_pairs_in_cohort,ineligible_pairs_in_cohort,observed_units,non_transitions_in_extract,extra_eligible_transitions_beyond_one_per_unit,availability_class
0,2023,405,404,404,0,401,4,3,RETROSPECTIVE_ONLY
1,2024,645,636,636,0,634,11,2,RETROSPECTIVE_ONLY
2,2025,856,779,779,0,773,83,6,RETROSPECTIVE_ONLY


## 2023 : surestimation de croissance

P1 réalisé **2,036102 %**, A/B full **2,870060 / 2,871314 %**. Occurrence réalisée **99,012 %**, supérieure aux deux estimations : elle n'explique pas la surestimation. Croissance conditionnelle **2,085372 %**, inférieure aux trois médianes historiques de l'estimation A/B. Saint-Elzear pèse 52,593 % avec croissance réalisée 1,969 %. Un ralentissement réalisé n'est pas une preuve de prévisibilité au cutoff.

## 2024 : explication de la pire erreur

La croissance conditionnelle réalisée est **1,699619 %**, contre **3,319802 / 3,244485 %** pour A/B. L'occurrence réalisée 98,295 % reste proche des 98,475 / 98,478 % prévus. Même la substitution de l'occurrence réalisée laisse P1 à **3,319802 / 3,244485 %** : le problème est principalement la croissance.

Saint-Elzear (33,333 % des candidates) et Le Carlyle (19,690 %) ont les P1 locaux les plus bas, **1,302539 / 1,473402 %**. Retirer l'un ou l'autre remonte la médiane globale d'environ **0,092 / 0,101 pp** seulement : plusieurs segments sont faibles, sans causalité ni somme d'effets identifiable. Deux chambres : 395 candidates, croissance conditionnelle 1,686 %.

Le mix des nouveaux segments Le Carlyle/The Met était reconstructible à D, mais les taux historiques mûrs des anciens bâtiments restaient élevés. La médiane Y−1 mûre **1,133244 %** portait sur seulement **13 transitions** sélectionnées par maturité. B améliore A de **0,033080 pp** sans segmenter les bâtiments ; sa hiérarchie d'occurrence augmente l'erreur par rapport à P0_G1. Le gain ne démontre donc pas une anticipation de la faiblesse 2024.

Le signal hypothétique à tester concerne l'hétérogénéité historique de croissance et la composition connue avec shrinkage/repli, pas les loyers faibles observés ensuite.

## 2025 : Saint-Elzear et compensation entre composantes

P1 réalisé **3,300878 %**, A/B **2,969328 / 2,924144 %**. Occurrence réalisée **90,303738 %**, croissance conditionnelle **3,716980 %** : occurrence surestimée, croissance sous-estimée.

Saint-Elzear concentre **65/83 non-transitions** sur **216 candidates**, taux réalisé **69,907407 %**. B y prévoit encore **98,277070 %**, contre A **98,189499 %** : aucun meilleur avertissement local. Avant la coupure, son taux qualifié était **98,005982 %** sur 1 003 labels, avec 216 en attente. La taille candidate est stable (215 → 216). Les 65 absences sont donc une explication rétrospective, pas un signal démontré prévisible.

L'extrait s'arrête en 2025 : les absences peuvent refléter retard ou omission, sans preuve de vacance. Les supports qualifiés récents des nouveaux bâtiments sont faibles ou nuls ; le repli est une limitation concrète de la hiérarchie.

In [6]:
special = buildings.loc[buildings.year.eq(2025) & buildings.segment.eq('Saint-Elzear')]
assert int(special.non_transitions.iloc[0]) == 65
assert int(buildings.loc[buildings.year.eq(2025), 'non_transitions'].sum()) == 83
display(special.round(6))

,year,dimension,segment,candidate_units,share_pct,observed_units,non_transitions,realized_occurrence_pct,realized_conditional_growth_pct,realized_segment_P1_pct,...,A_occurrence_pct,A_growth_pct,A_segment_P1_pct,A_occurrence_error_count,P1_G1_occurrence_pct,P1_G1_growth_pct,P1_G1_segment_P1_pct,P1_G1_occurrence_error_count,P1_without_building_pct,leave_out_sensitivity_pp
104,2025,building,Saint-Elzear,216,25.233645,151,65,69.907407,4.195614,0.912654,...,98.189499,3.024079,2.969328,61.089318,98.27707,2.931619,2.924196,61.278472,3.481768,0.180889


## Économie locative : baux mûrs seulement

sRent, sRentEffective, gap source et flag sConcession sont agrégés seulement pour les baux signés, commencés et terminés à D. Aucune reconstruction PromoPay, aucune inférence de récurrence. Les caractéristiques d'effectif/gap restent hors features autorisées et de disponibilité stricte **UNKNOWN**. Les séries asking non qualifiées ne sont pas jointes.

In [7]:
display(findings['rent_economics'].round(4))

,year,dimension,segment,mature_leases,valid_gap,contractual_rent_median,effective_rent_median,gap_median_pct,concession_flag_known,concession_flag_pct,availability_class
0,2023,sBuilding,Daniel-Johnson,170,170,1935.0,1890.375,0.0000,170,40.0000,UNKNOWN
1,2023,sBuilding,Levesque,315,315,1415.0,1395.000,0.0000,315,47.3016,UNKNOWN
2,2023,sBuilding,Saint-Elzear,847,847,1885.0,1840.280,0.0000,847,48.6423,UNKNOWN
3,2023,_start_year,2017,93,93,1815.0,1800.000,0.0000,93,38.7097,UNKNOWN
4,2023,_start_year,2018,172,172,1610.0,1584.580,0.0000,172,38.3721,UNKNOWN
5,2023,_start_year,2019,328,328,1730.0,1694.835,0.0000,328,44.2073,UNKNOWN
6,2023,_start_year,2020,373,373,1775.0,1735.000,0.0000,373,48.7936,UNKNOWN
7,2023,_start_year,2021,349,349,1850.0,1783.810,4.0944,349,54.4413,UNKNOWN
8,2023,_start_year,2022,17,17,1700.0,1700.000,4.9121,17,58.8235,UNKNOWN
9,2024,sBuilding,Daniel-Johnson,291,291,1935.0,1888.670,0.0000,291,44.6735,UNKNOWN


## Signaux prévisibles, rétrospectifs ou inconnus

Le calendrier est démontré disponible. Les champs CRM respectent les dates du contrat mais leur version historique est inconnue ; cette réserve est conservée. Toute explication **RETROSPECTIVE_ONLY** reste ex post, scénario ou limitation, jamais feature.

In [8]:
display(findings['signals'])

,signal,year,observed_effect,availability_class,evidence,potential_model_use
0,Calendrier de coupure et année prédite,2023,2022-12-31,PREDICTABLE_AT_CUTOFF,"Calendrier déterministe, indépendant des donné...",Calendrier autorisé.
1,Composition et échéances reconstruites,2023,405 candidates,UNKNOWN,Contrat daté respecté ; fidélité des versions ...,Admissible selon contrat ; confirmer la fidéli...
2,Historique qualifié d’occurrence et de croissance,2023,"p A=98.328 %, g A=2.919 %",UNKNOWN,Dates de maturité <= D ; aucun journal des ver...,Statistiques déjà utilisées sous réserve contr...
3,Gap contractuel/effectif des baux mûrs,2023,Voir économie locative agrégée à D.,UNKNOWN,Baux terminés <= D ; ancien effectif/gap non q...,Diagnostic seulement ; ne pas réintroduire le ...
4,Taux de transition de Y,2023,99.012 %,RETROSPECTIVE_ONLY,Label révélé après les prédictions de Y.,Explication ex post uniquement.
5,Croissance conditionnelle de Y,2023,2.085 %,RETROSPECTIVE_ONLY,Loyers des nouveaux baux et intervalle réalisé...,Explication ex post uniquement.
6,Mix réalisé renouvellement/relocation,2023,Non utilisé par A/B ; statut futur interdit.,RETROSPECTIVE_ONLY,sRenewal du nouveau bail appartient aux résult...,"Scénario ou diagnostic, jamais feature."
7,Calendrier de coupure et année prédite,2024,2023-12-31,PREDICTABLE_AT_CUTOFF,"Calendrier déterministe, indépendant des donné...",Calendrier autorisé.
8,Composition et échéances reconstruites,2024,645 candidates,UNKNOWN,Contrat daté respecté ; fidélité des versions ...,Admissible selon contrat ; confirmer la fidéli...
9,Historique qualifié d’occurrence et de croissance,2024,"p A=98.475 %, g A=3.320 %",UNKNOWN,Dates de maturité <= D ; aucun journal des ver...,Statistiques déjà utilisées sous réserve contr...


## Externe existant : aucune disponibilité forcée

Zéro signal externe admissible aux trois backtests. TAL legacy 2025 : seulement neuf lignes admissibles à l'origine 2026, utilité/applicabilité non acquise. Nouveaux paramètres TAL 2026 : publication après l'origine 2026, donc non pertinents pour ce périmètre temporel. IPC/SCHL/Ontario : historiques non qualifiés. Aucun transfert de règles QC/ON ni nouvelle collecte.

In [9]:
display(findings['external'])
assert findings['external'][['eligible_2023','eligible_2024','eligible_2025']].to_numpy().sum() == 0

,signal,source,rows,values_present,eligible_2023,eligible_2024,eligible_2025,eligible_2026,classification,limitation
0,average_rent,SCHL,2,2,0,0,0,0,NOT_HISTORICALLY_QUALIFIED,Version/date historique non qualifiée ; exclus...
1,average_rent_growth,SCHL,14,2,0,0,0,0,NOT_HISTORICALLY_QUALIFIED,Version/date historique non qualifiée ; exclus...
2,non_turnover_rent_growth,SCHL,2,2,0,0,0,0,NOT_HISTORICALLY_QUALIFIED,Version/date historique non qualifiée ; exclus...
3,ontario_guideline,Gouvernement de l’Ontario,10,10,0,0,0,0,NOT_HISTORICALLY_QUALIFIED,Version/date historique non qualifiée ; exclus...
4,rent_cpi_annual_growth,Statistique Canada,18,18,0,0,0,0,NOT_HISTORICALLY_QUALIFIED,Version/date historique non qualifiée ; exclus...
5,rent_cpi_annual_mean,Statistique Canada,18,18,0,0,0,0,NOT_HISTORICALLY_QUALIFIED,Version/date historique non qualifiée ; exclus...
6,rent_cpi_ytd_growth,Statistique Canada,2,2,0,0,0,0,NOT_HISTORICALLY_QUALIFIED,Version/date historique non qualifiée ; exclus...
7,rent_cpi_ytd_mean,Statistique Canada,2,2,0,0,0,0,NOT_HISTORICALLY_QUALIFIED,Version/date historique non qualifiée ; exclus...
8,tal_legacy_building_services,TAL,2,2,0,0,0,1,SAFE_BUT_WEAK,Applicabilité et utilité prédictive à confirme...
9,tal_legacy_capital_expenditure,TAL,2,2,0,0,0,1,SAFE_BUT_WEAK,Applicabilité et utilité prédictive à confirme...


## Cohérence du périmètre 2026, sans forecast

931 candidates : Québec 810, Ontario 121. La composition sert seulement à vérifier que les risques de support/repli restent pertinents. Aucun label ni forecast 2026 ne sert à choisir une méthode.

In [10]:
cohort = findings['cohort_2026']
assert cohort.loc[cohort.feature.eq('building'), 'units'].sum() == 931
display(cohort.round(4))

,year,feature,category,units,share_pct
0,2026,building,Saint-Elzear,268,28.7863
1,2026,building,Le Carlyle,180,19.3340
2,2026,building,Westpark,159,17.0784
3,2026,building,Daniel-Johnson,128,13.7487
4,2026,building,The Met,121,12.9968
5,2026,building,Levesque,75,8.0559
6,2026,province,Quebec,810,87.0032
7,2026,province,Ontario,121,12.9968
8,2026,expiry_month,6,143,15.3598
9,2026,expiry_month,9,108,11.6004


## Trois hypothèses et prochaine action unique

1. **Croissance bâtiment → global sur unités/années positives P1** : conserver p de A et le contrat ; support et shrinkage 10/10 existants fixés avant test. Signal historique conditionnellement disponible ; risque d'overfit modéré. Rejeter si pire erreur/biais augmente ou si le gain dépend de 2024 en dégradant 2025.
2. **Occurrence bâtiment → global** : tester calibration locale et support/en attente ; ne pas prétendre anticiper les 65 absences futures. Rejeter sans gain Brier/calibration et sans maîtrise de l'erreur P1.
3. **Croissance récente mûre rétrécie** : supports 7/13/28, forte sélection de maturité ; aucune fenêtre ni poids optimisé. Rejeter si le bénéfice 2024 masque une dégradation 2025 ou un biais de sélection.

**Recommandation unique : D, tester le candidat 1.** Hypothèse falsifiable, aucun gain annoncé, aucun modèle construit, aucun champion final sélectionné. A/B full partagent les mêmes signes d'erreur ; 50/50 n'améliore pas A en MAE. P1_G0 gagne les métriques agrégées mais manque davantage 2025. Le détail des signaux, risques et critères est dans le rapport [error_forensics.md](../docs/error_forensics.md).